# 03 · Qwen translation baseline

Context-aware model: give Qwen a Chinese post, get English back. Provide slang definitions.

Before running (Colab):
1. Change runtime to L4.
2. Upload data (I added this to the code because I kept forgetting to add it)


## 1. Install and import

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files

uploaded = files.upload() # choose rq1_matched_eval.csv AND rq1_matched_test.csv
print(list(uploaded.keys())) # should print both file names

Saving rq1_matched_eval.csv to rq1_matched_eval.csv
Saving rq1_matched_test.csv to rq1_matched_test.csv
['rq1_matched_eval.csv', 'rq1_matched_test.csv']


In [ ]:
# src for using a HF model (plus Assignment 3): https://medium.com/data-science/choosing-and-implementing-hugging-face-models-026d71426fbe$0
%pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 144.3 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1


In [ ]:
import time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Check that I'm using the right GPU
print(torch.cuda.get_device_name(0))

NVIDIA L4


## 2. Load the model

In [ ]:
#model: https://huggingface.co/Qwen/Qwen2.5-7B-Instruct$0
model_name = "Qwen/Qwen2.5-7B-Instruct"

# Structure of this code is based on Assignment 3 & the model's Quickstart guide
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = model = AutoModelForCausalLM.from_pretrained( #parameters from the Quickstart documentation
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

#make sure it loaded on the right device
print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded on: cuda:0


## 3. Helper function: translate one post

Since Qwen is a chat model, we need to send it a system and user message:
- system message saying what its job is
- user message containing the post

In [ ]:
base_prompt = (
    "You are a professional translator. "
    "Translate the user's Chinese social media post into English. "
    "Keep the same tone and emotion as the original. "
    "Output only the English translation, with no notes or explanations."
)

# The winning variant from the blind comparison: the slang glossary goes in the USER message,
# before the post. (This wording must match what the judges saw in the comparison.)


def format_definitions(definitions_text):
    # The csv stores several definitions in one cell, separated by " || "
    # Turn that into one bullet line per term:
    lines = []
    for one_definition in str(definitions_text).split(" || "):
        lines.append("- " + one_definition.strip())
    return "\n".join(lines)


def build_messages(chinese_text, definitions_text):
    # Returns the messages to send to Qwen for one post.
    glossary = format_definitions(definitions_text)

    system_text = (
        base_prompt + " "
        "The message begins with a slang glossary. Use it to understand the post if slang is included, "
        "but translate only the post."
    )
    user_text = "Slang glossary:\n" + glossary + "\n\nPost:\n" + chinese_text

    return [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]

In [ ]:
# This is the same function as before essentially, but I just had to edit the input because of the way I wrote my variants
# I wrote out the whole role: xyz name: xyz in the return of the functionabove because it made the function above cleaner

def translate_messages(messages):
    # turn the conversation into Qwen's expected text format
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # turn the text into tokens & move to the same device as the model (so it can run quickly)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    # generate the answer
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

    # keep only the new tokens (the answer) and turn them back into text
    prompt_length = inputs["input_ids"].shape[1]
    answer_tokens = output_tokens[0][prompt_length:]
    english_text = tokenizer.decode(answer_tokens, skip_special_tokens=True) #only decode the answer tokens (not the prompt)

    return english_text.strip()

In [ ]:
import os

save_folder = "/content/drive/MyDrive/ISchool/MIDS/266"
save_every = 50   # save a backup every 50 posts in case Colab crashes


# Helper function to translate the SLANG posts in one matched file
# definitions_column = which column holds the definitions to give Qwen ("slang_definitions" for RQ2, "incorrect_slang_definitions" for RQ3)
# condition_name = the label saved in the "condition" column
def translate_matched_file(input_file, output_file, definitions_column, condition_name):

    # Load the posts
    matched = pd.read_csv(input_file)

    # Keep only the slang posts. Control posts have no definitions,
    # so their translation would be identical to the baseline.
    slang_posts = matched[matched["role"] == "slang"]

    # Translate each post, one at a time
    results = [] # one dictionary per post
    count = 0 # how many posts are done so far

    for index, row in slang_posts.iterrows():
        # Build the conversation (glossary + post), then translate it
        messages = build_messages(row["text"], row[definitions_column])
        english = translate_messages(messages)
        count = count + 1

        # Copy every column from the matched file (pair_id, role, label, ...)
        result_row = {}
        for column_name in matched.columns:
            result_row[column_name] = row[column_name]

        # Add our two new columns
        result_row["condition"] = condition_name
        result_row["translation"] = english

        results.append(result_row)

        # Every 50 posts, save a backup and print progress
        if count % save_every == 0:
            pd.DataFrame(results).to_csv(output_file, index=False, encoding="utf-8-sig")

    # Final save, with all posts
    pd.DataFrame(results).to_csv(output_file, index=False, encoding="utf-8-sig")

In [ ]:
# Run the context-aware (correct definitions) translations on eval and test (RQ2)
start = time.time()

eval_output = os.path.join(save_folder, "rq2_matched_eval_context_aware.csv")
translate_matched_file("rq1_matched_eval.csv", eval_output, "slang_definitions", "context-aware")

test_output = os.path.join(save_folder, "rq2_matched_test_context_aware.csv")
translate_matched_file("rq1_matched_test.csv", test_output, "slang_definitions", "context-aware")

minutes = (time.time() - start)/60
print("All done in", round(minutes), "minutes")

All done in 8 minutes


In [ ]:
# Run the incorrect-definitions translations on eval and test (RQ3)
start = time.time()

eval_output = os.path.join(save_folder, "rq3_matched_eval_incorrect.csv")
translate_matched_file("rq1_matched_eval.csv", eval_output, "incorrect_slang_definitions", "incorrect")

test_output = os.path.join(save_folder, "rq3_matched_test_incorrect.csv")
translate_matched_file("rq1_matched_test.csv", test_output, "incorrect_slang_definitions", "incorrect")

minutes = (time.time() - start) / 60
print("All done in", round(minutes), "minutes")

All done in 8 minutes
